In [15]:
# Tools
from toolbox_langchain import ToolboxClient

In [16]:
TOOLBOX_URL = "http://127.0.0.1:5000"
client = ToolboxClient(
    TOOLBOX_URL, 
    # client_headers={"Authorization": auth_token_provider}
)

In [17]:
tools = await client.aload_toolset("cymbal_air")

ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fdba740cc50>


In [18]:
len(tools)

7

In [19]:
from langchain_core.prompts import ChatPromptTemplate
import asyncio
import os
import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Sequence
from pytz import timezone

In [20]:
PREFIX = """The Cymbal Air Customer Service Assistant helps customers of Cymbal Air with their travel needs.

Cymbal Air (airline unique two letter identifier as CY) is a passenger airline offering convenient flights to many cities around the world from its
hub in San Francisco. Cymbal Air takes pride in using the latest technology to offer the best customer
service!

Cymbal Air Customer Service Assistant (or just "Assistant" for short) is designed to assist
with a wide range of tasks, from answering simple questions to complex multi-query questions that
require passing results from one query to another. Using the latest AI models, Assistant is able to
generate human-like text based on the input it receives, allowing it to engage in natural-sounding
conversations and provide responses that are coherent and relevant to the topic at hand. The assistant should 
not answer questions about other people's information for privacy reasons. 

Assistant is a powerful tool that can help answer a wide range of questions pertaining to travel on Cymbal Air
as well as amenities of San Francisco Airport."""

SUFFIX = """Begin! Use tools if necessary. Respond directly if appropriate."""

In [21]:
def get_datetime():
    formatter = "%A, %m/%d/%Y, %H:%M:%S"
    now = datetime.now(timezone("US/Pacific"))
    return now.strftime(formatter)
        
def create_prompt_template() -> ChatPromptTemplate:
    current_datetime = "Today's date and current time is {cur_datetime}."
    template = "\n\n".join(
        [
            PREFIX,
            current_datetime,
            SUFFIX,
        ]
    )
    prompt = ChatPromptTemplate.from_messages(
        [("system", template), ("placeholder", "{messages}")]
    )
    prompt = prompt.partial(cur_datetime=get_datetime)
    return prompt

In [22]:
prompt = create_prompt_template()

In [23]:
# prompt

In [24]:
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    ToolCall,
    ToolMessage,
)

### Gemini

In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(
    max_output_tokens=512,
    model="gemini-2.5-flash",
    temperature=0.0,
    api_key=os.environ["GEMINI_API_KEY"]
)

In [8]:
model_with_tools = model.bind_tools(tools)

In [18]:
model_runnable = prompt | model_with_tools

In [26]:
messages = [
    AIMessage(content="Welcome to Cymbal Air!  How may I assist you?"),
    HumanMessage(content="When is the next flight to Los Angeles?"),
]
res = await model_runnable.ainvoke({"messages": messages})

In [29]:
res.content[0]

{'type': 'text',
 'text': 'I can help you with that! What date would you like to fly to Los Angeles? Please use the YYYY-MM-DD format.',
 'extras': {'signature': 'CqsNAQw51sdMVKHYpqRVjbAuwHSE9+xHsMeV9fe1PwB+pvJRiHzUF+nK8UXp4GgFM4QBVo0KjZQCk7sM0bavFVl9KaJTBn9H+QJw29lI+9LqB/ktGDubkDaGKHmXtRCT7j/WNAcuz4VWRiSFmLvUyby5BA4kltkSMoYv437jHCGKo3fO2aYrcztMtYy3WmVjE7kuiPOswYafXSs0ThnEq0uCEjv7wEPDUzAjxtW1VX20S37yugCBSCxzS2if3SX6kZII+8vThKTaEMFyDZxjXH1lEQymr5/Wo+MsKAjhqWYprvsd3eaODX79cTlIutF9g8uQ5p5p1wDK4YbpXwYnoATVOfIBi2LOutv/E0mOOYQGZSqwsHzVg76Z+HqAqV2HnTkUpvh4mG+8gubdwvf3kHdgbak6ODtvODw/dOLaC1/VCZRdchLskp6xmAPK+LMmTCBCxHgO/9iaviRGjduqjmKnB7MgAGjvl7clapxAMS9F8IemJsI35J4rlZ0vkSCOd/gT8cf99CdQIJz+dlw9mjjY0uYO5nx1ENUjqPM6jFpyUy4cHLpAonRFH8YoAzY2vlnQPJhGHsrdUc/kXKDccBwnE9WTZ/ZMKD4IhULarcVTzHVpuos0JQWtxsazIbb1qPd/CFGOFuK1LbzXjFsuyIxlAX05m8gy0EqCa+T6+6hwfy3C3uBun34jQY5JLmsZ2FkEtSydchye3giAzgIn7NWSu/uJQfVB5bNl3l4INn3zCReftGf9M/GcCuQTSaEHAEh0LytI0rmRDfMnXcrLPI5SAOywiri4F3XDcGexZ4wzyisn/9teqy9xVW4AuK++00fJF

### Groq

In [28]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

True

In [33]:
model = ChatOpenAI(
    # max_output_tokens=512,
    model_name=f"openai/gpt-oss-120b",
    temperature=0,
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=os.environ.get("GROQ_API_KEY"),
    # top_p=1,
    # max_retries=3,
    # request_timeout=60,
)

In [42]:
model_with_tools = model.bind_tools(tools)

In [43]:
model_runnable = prompt | model_with_tools

In [44]:
messages = [
    AIMessage(content="Welcome to Cymbal Air!  How may I assist you?"),
    HumanMessage(content="When is the next flight to Los Angeles?"),
]
res = await model_runnable.ainvoke({"messages": messages})

In [45]:
res.__dict__

{'content': 'Sure thing! Could you let me know which date you’d like to travel (e.g.,\u202f2026‑05‑24) so I can find the next available Cymbal\u202fAir flight from San\u202fFrancisco (SFO) to Los\u202fAngeles (LAX) for you?',
 'additional_kwargs': {'refusal': None},
 'response_metadata': {'token_usage': {'completion_tokens': 178,
   'prompt_tokens': 1343,
   'total_tokens': 1521,
   'completion_tokens_details': {'accepted_prediction_tokens': None,
    'audio_tokens': None,
    'reasoning_tokens': 109,
    'rejected_prediction_tokens': None},
   'prompt_tokens_details': None,
   'queue_time': 0.053695391,
   'prompt_time': 0.065980669,
   'completion_time': 0.38788955,
   'total_time': 0.453870219},
  'model_provider': 'openai',
  'model_name': 'openai/gpt-oss-120b',
  'system_fingerprint': 'fp_bc9a6c9b5f',
  'id': 'chatcmpl-41c990dd-b0d8-4e8c-afb3-b701e05b8dfb',
  'service_tier': 'on_demand',
  'finish_reason': 'stop',
  'logprobs': None},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--

In [46]:
# With Config
config = {
    "configurable": {
        "thread_id": "test-id",
        "checkpoint_ns": "",
        # "auth_token_getters": {
        #     "my_google_service": lambda: get_user_id_token()
        # },
    },
}

In [47]:
messages = [
    AIMessage(content="Welcome to Cymbal Air!  How may I assist you?"),
    HumanMessage(content="When is the next flight to Los Angeles?"),
]
res = await model_runnable.ainvoke({"messages": messages}, config)

In [48]:
res.__dict__

{'content': '',
 'additional_kwargs': {'refusal': None},
 'response_metadata': {'token_usage': {'completion_tokens': 292,
   'prompt_tokens': 1343,
   'total_tokens': 1635,
   'completion_tokens_details': {'accepted_prediction_tokens': None,
    'audio_tokens': None,
    'reasoning_tokens': 239,
    'rejected_prediction_tokens': None},
   'prompt_tokens_details': None,
   'queue_time': 0.053769647,
   'prompt_time': 0.067380902,
   'completion_time': 0.617526115,
   'total_time': 0.684907017},
  'model_provider': 'openai',
  'model_name': 'openai/gpt-oss-120b',
  'system_fingerprint': 'fp_8a618bed98',
  'id': 'chatcmpl-f532487b-2a3d-4063-b1f6-5cad73f281e1',
  'service_tier': 'on_demand',
  'finish_reason': 'tool_calls',
  'logprobs': None},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--019e53d7-d30a-78a3-9c2b-131a394c7f05-0',
 'tool_calls': [{'name': 'list_flights',
   'args': {'arrival_airport': 'LAX',
    'date': '2026-05-23',
    'departure_airport': 'SFO'},
   'id': 'fc_cacfb6b6-67